# Ingestion report

Summarise what is already on disk for a **data lake deployment**: catalogs (Parquet/HATS), spectra and cutouts (Zarr tiles), optional DESI batch-ingest checkpoints, and cross-match tables.

**Configuration** is resolved like the `dl-*` CLIs: optional explicit path in the next cell, else `$DATA_LAKE_CONFIG`, else walk upward from the current working directory for `lake_config.toml`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import polars as pl
from IPython.display import display
import zarr

from data_lake.config import LakeConfig, LakeConfigNotFound

# Set to a concrete path to override discovery (otherwise None)
LAKE_CONFIG_TOML: Path | None = None

# If True, open every spectrum/cutout tile to sum source counts (accurate but I/O heavy for huge lakes)
COUNT_ZARR_ROWS: bool = True

try:
    cfg = LakeConfig.discover(LAKE_CONFIG_TOML)
except LakeConfigNotFound as e:
    raise RuntimeError(
        "No lake_config.toml found. Create one with dl-init, export DATA_LAKE_CONFIG, "
        "or set LAKE_CONFIG_TOML in this cell."
    ) from e

ROOT = cfg.lake.root
print("Lake:", cfg.lake.name)
print("Root:", ROOT)
print("Config:", cfg.source_path)
print("HEALPix order (default):", cfg.partitioning.hats_order)

## Helpers

In [ ]:
def _safe_json(path: Path) -> dict | None:
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None


def _child_survey_dirs(layer_root: Path) -> list[Path]:
    """Immediate subdirectories that look like a survey (contain tiles or a *info.json)."""
    if not layer_root.is_dir():
        return []
    out: list[Path] = []
    for p in sorted(layer_root.iterdir()):
        if not p.is_dir():
            continue
        if p.name == "crossmatch":
            continue
        marker = (
            any(p.glob("catalog_info.json"))
            or any(p.glob("spectrum_info.json"))
            or any(p.glob("cutout_info.json"))
            or any(p.glob("Norder=*"))
        )
        if marker:
            out.append(p)
    return out


def _count_parquet_tiles(survey_dir: Path) -> int:
    return sum(1 for f in survey_dir.rglob("*.parquet") if f.name.startswith("Npix="))


def _iter_zarr_tiles(survey_dir: Path) -> list[Path]:
    return sorted(survey_dir.rglob("*.zarr"))


def _zarr_n_sources(tile: Path) -> int:
    store = zarr.storage.LocalStore(str(tile))
    g = zarr.open_group(store=store, mode="r", zarr_format=3)
    return int(g["source_id"].shape[0])


def _crossmatch_surveys(cross_root: Path) -> list[Path]:
    if not cross_root.is_dir():
        return []
    return sorted(p for p in cross_root.iterdir() if p.is_dir())

## Catalogs (Parquet / HATS)

In [ ]:
rows = []
cat_root = cfg.catalogs_root
for survey in _child_survey_dirs(cat_root):
    info = _safe_json(survey / "catalog_info.json") or {}
    rows.append(
        {
            "survey": survey.name,
            "hats_order": info.get("hats_order"),
            "total_rows": info.get("total_rows"),
            "total_columns": info.get("total_columns"),
            "n_parquet_tiles": _count_parquet_tiles(survey),
            "path": str(survey),
        }
    )

if rows:
    display(pl.DataFrame(rows))
else:
    print(f"No catalog surveys under {cat_root}")

## Spectra (Zarr tiles)

In [ ]:
spec_rows = []
spec_root = cfg.spectra_root
for survey in _child_survey_dirs(spec_root):
    info = _safe_json(survey / "spectrum_info.json") or {}
    tiles = _iter_zarr_tiles(survey)
    n_sources = None
    if COUNT_ZARR_ROWS and tiles:
        n_sources = sum(_zarr_n_sources(t) for t in tiles)
    ck = survey / ".ingest_checkpoint.json"
    n_ck = None
    if ck.is_file():
        data = _safe_json(ck) or {}
        n_ck = len(data.get("completed", []))
    fail = survey / ".ingest_failures.jsonl"
    n_fail = sum(1 for _ in fail.open()) if fail.is_file() else 0
    spec_rows.append(
        {
            "survey": survey.name,
            "hats_order": info.get("hats_order"),
            "n_zarr_tiles": len(tiles),
            "n_spectra_in_zarr": n_sources,
            "wavelength_mode": info.get("wavelength_mode"),
            "has_resolution": info.get("has_resolution"),
            "checkpoint_completed_files": n_ck,
            "ingest_failure_lines": n_fail,
            "path": str(survey),
        }
    )

if spec_rows:
    display(pl.DataFrame(spec_rows))
else:
    print(f"No spectrum surveys under {spec_root}")

if not COUNT_ZARR_ROWS:
    print("Note: COUNT_ZARR_ROWS=False — enable it to sum spectra from each tile's source_id length.")

## Cutouts (Zarr tiles)

In [ ]:
cut_rows = []
cut_root = cfg.cutouts_root
for survey in _child_survey_dirs(cut_root):
    info = _safe_json(survey / "cutout_info.json") or {}
    tiles = _iter_zarr_tiles(survey)
    n_sources = None
    if COUNT_ZARR_ROWS and tiles:
        n_sources = sum(_zarr_n_sources(t) for t in tiles)
    cut_rows.append(
        {
            "survey": survey.name,
            "hats_order": info.get("hats_order"),
            "n_bands": info.get("n_bands"),
            "height": info.get("height"),
            "width": info.get("width"),
            "n_zarr_tiles": len(tiles),
            "n_cutouts_in_zarr": n_sources,
            "path": str(survey),
        }
    )

if cut_rows:
    display(pl.DataFrame(cut_rows))
else:
    print(f"No cutout surveys under {cut_root}")

## Cross-matches (`catalogs/crossmatch/`)

In [ ]:
xm_root = cat_root / "crossmatch"
xm_rows = []
for xm in _crossmatch_surveys(xm_root):
    info = _safe_json(xm / "catalog_info.json") or {}
    xm_rows.append(
        {
            "name": xm.name,
            "survey_a": info.get("survey_a"),
            "survey_b": info.get("survey_b"),
            "match_radius_arcsec": info.get("match_radius_arcsec"),
            "hats_order": info.get("hats_order"),
            "n_parquet_tiles": _count_parquet_tiles(xm),
            "path": str(xm),
        }
    )

if xm_rows:
    display(pl.DataFrame(xm_rows))
else:
    print(f"No cross-match tables under {xm_root}")

## Catalog ↔ spectra linkage (optional)

If the catalog has `_spectrum_index`, count how many rows point into an ingested spectrum tile (`_spectrum_index >= 0`). Requires DuckDB and can scan the full catalog.

In [ ]:
RUN_LINKAGE_STATS = False

if RUN_LINKAGE_STATS:
    from data_lake.io.catalog import CatalogAccessor

    link_rows = []
    for survey in _child_survey_dirs(cat_root):
        try:
            acc = CatalogAccessor(ROOT, survey.name)
        except Exception as exc:
            link_rows.append({"survey": survey.name, "error": str(exc)})
            continue
        cols = acc.columns
        if "_spectrum_index" not in cols:
            link_rows.append({"survey": survey.name, "with_spectrum_index": False})
            continue
        total = int(acc.query("SELECT COUNT(*) AS n FROM catalog")["n"][0])
        with_spec = int(
            acc.query(
                "SELECT COUNT(*) AS n FROM catalog WHERE _spectrum_index >= 0"
            )["n"][0]
        )
        link_rows.append(
            {
                "survey": survey.name,
                "catalog_rows": int(total),
                "rows_with_spectrum": int(with_spec),
                "fraction": float(with_spec) / int(total) if int(total) else None,
            }
        )
    display(pl.DataFrame(link_rows))
else:
    print("Set RUN_LINKAGE_STATS=True to compute _spectrum_index coverage per catalog.")